In [13]:
# some imports 
import numpy as np 
import networkx as nx
import copy 
from itertools import product
from collections import Counter
import matplotlib.pyplot as plt
from numpy import random
#D-Wave Ocean SDK related
import dimod 
from dwave.system import DWaveSampler
from dwave.samplers import SimulatedAnnealingSampler
# from dwave.cloud import Client
# client = Client.from_config()
from dwave.embedding import embed_ising, unembed_sampleset
from minorminer import find_embedding
import pickle
from pathlib import Path

seed = 1224
random.seed(seed)



In [ ]:
embeddings = pickle.loads(Path("../pkl/embeddings.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("../pkl/SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("../pkl/QA_sample_sets.pkl").read_bytes())
sa_results = pickle.loads(Path("../pkl/sa_results.pkl").read_bytes())
qa_results = pickle.loads(Path("../pkl/qa_results.pkl").read_bytes())
graphs = pickle.loads(Path("../pkl/graphs.pkl").read_bytes())
labels = pickle.loads(Path("../pkl/labels.pkl").read_bytes())

In [15]:
from dwave.samplers import SteepestDescentSolver

SD_solver = SteepestDescentSolver()

def run_SD(graph, SA_sampleset, label):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")

    SD_sampleset = SD_solver.sample_ising(
        h, J,
        initial_states=SA_sampleset,
        num_reads=len(SA_sampleset),
    )

    SA_energies = SA_sampleset.data_vectors["energy"]
    SD_energies = SD_sampleset.data_vectors["energy"]
    return SD_sampleset, {
        "label": label,
        "SA_expected": np.mean(SA_energies),
        "SA_best": SA_energies.min(),
        "SD_expected": np.mean(SD_energies),
        "SD_best": SD_energies.min(),
    }

SD_sample_sets = []
SD_summaries = []
for graph, SA_ss, label in zip(graphs, SA_sample_sets, labels):
    ss, summary = run_SD(graph, SA_ss, label)
    SD_sample_sets.append(ss)
    SD_summaries.append(summary)